In [ ]:
INPUT_ROOT = "/kaggle/input/your-keyframes-dataset"


# PaddleOCR CPU/GPU backend gate — self-contained v5.2 / sidecar v1

1. **Import this notebook** into Kaggle.
2. **Attach your keyframes Dataset**. No external code ZIP or manifest is needed.
3. **Enable Internet** for the pinned binary wheels and revision-pinned models. For GPU, also set **Accelerator = GPU (T4/P100)** in the Kaggle session options.
4. **Edit only `INPUT_ROOT`** in cell 1, then **Run All**; download ZIP + `.sha256` at the end.

The device is chosen automatically (see v5.2 below). CPU profile: Python 3.13 keeps NumPy 2.2.6, PaddlePaddle 3.3.1,
PaddleOCR/PaddleX 3.7.0; 3.10–3.12 keeps its existing NumPy 1.26.4 profile.
The pip-less isolated venv uses host pip `--python` (no ensurepip, `-S`,
global install, NumPy unpinning or source builds). Eight allowlisted source,
config and schema files are embedded with per-file and payload SHA256.
The unchanged shared contract implementation is embedded too; attached v3/v4
packages are never discovered or loaded. No data, credentials or weights embedded.

**v4 failure:** CPU oneDNN/PIR raised `ConvertPirAttribute2RuntimeAttribute`
on `pir::ArrayAttribute<pir::DoubleAttribute>`. v5 does not assume an env var fixes it.
In the exact installed stack it first verifies whether the constructor/common-args
API accepts `enable_mkldnn=False`, then runs the real model on the first existing
image in deterministic canonical order, with the same 32 px padding and adapter.
Only if Paddle's `get_flags` confirms `FLAGS_use_mkldnn` exists may a **fresh child**
try `FLAGS_use_mkldnn=0` before importing paddle. Effective flags, API/source evidence,
model hashes and full bounded tracebacks are logged. `FLAGS_enable_pir_api=0` is
**not attempted**, because compatibility for this stack has not been verified.

No output/checkpoint is created until the mandatory inference smoke passes.
If no candidate passes, stop and inspect `outputs/diagnostics-v5/`: no thousands of
backend error records. A later backend inference failure also stops immediately.
An individual corrupt image after the smoke remains `error`; a corrupt first image
stops smoke and must be corrected. A `sitecustomize`/`wrapt` startup warning alone
is neither success nor failure; do not install wrapt globally. Pip check and import
smoke remain separate from inference success. No new wrapt dependency is guessed.

This v5 is **offline/static/mock validated only, not Kaggle inference verified**.
The runtime API checks and smoke determine whether a workaround is safe on your session.
No accuracy, ABI or real-inference success claim is made by this notebook.

Supported layouts: video root `L99_V003/1.jpg`, group root `L99/L99_V003/1.jpg`,
or a folder of videos/groups; filenames must be numeric JPEG/PNG. Optional per-video
`mapping.csv`: `n,frame_idx,pts_time`; keyframe number is NOT frame index.
White padding 32 px per side, no wrapper resize, exact inverse and separate clipped
coordinates. Preserve `ok/empty/missing/error/unknown`; complete only without
error/missing/unknown. Join only on `(video_id,keyframe_n,image_sha256)`.

Outputs: `/kaggle/working/kis-kaggle-ocr-v5/<payload_hash>/outputs/ocr-v1`.
Run All again resumes only matching source/config/input/model/backend hashes; no
v3/v4 output/resume reuse, no implicit error retry. Future explicit retries use
`--resume --retry-errors`. No caption, production, FAISS or Milvus writes.
Optional offline caches retain the existing `ocr-wheelhouse` and revision/hash-locked
`ocr-models` conventions. Normal keyframes-only flow downloads models automatically.
Hashes protect corruption, not a maliciously replaced notebook; import a trusted copy.

**v5.1 fix:** the v5 API probe called PaddleOCR 3.7.0's `parse_common_args` without its required keyword-only `default_enable_hpi`, so every run failed with `BACKEND_API_UNAVAILABLE` before inference. The probe now passes the argument the installed parser declares.

**v5.2 GPU mode:** if `nvidia-smi` sees a GPU, the notebook installs `paddlepaddle-gpu` (3.3.1 preferred, otherwise the newest 3.2–3.3 wheel the index offers for this Python) from Paddle's own CUDA index (`cu126` when the driver is >= 550.54.14, else `cu118`), into the same isolated venv, and runs on `gpu:0`. All other pins are unchanged and the install stays binary-only. Without a GPU it behaves exactly like v5.1 on CPU. If the GPU wheel cannot be installed, or the one-image GPU smoke fails with a CUDA/GPU error, the notebook says so, keeps the evidence in `outputs/diagnostics-v5-gpu-failed/` and rebuilds a CPU venv. Force CPU with `os.environ['KIS_OCR_DEVICE'] = 'cpu'` before the dependency cell. GPU and CPU runs have different config hashes, so they never resume each other. The runtime venv (several GB with the GPU wheel) is created under `/kaggle/temp` (or `/tmp`), not `/kaggle/working`, so it is not saved with the output; it falls back to `/kaggle/working` only if no scratch directory with enough free space is found. GPU mode is mock/static validated only, **not verified on a real Kaggle GPU**.


In [ ]:
from pathlib import Path, PurePosixPath
import base64
import hashlib
import importlib.util
import json
import os
import re
import stat
import subprocess
import sys
import zipfile

MAX_PAYLOAD_BYTES = 1024 * 1024
ALLOWED_FILES = {
    'tools/kaggle_ocr.py',
    'tools/kaggle_ocr_v5.py',
    'config/ocr-kaggle-py313-v5.json',
    'config/ocr-kaggle.json',
    'config/ocr-kaggle-py313.json',
    'config/requirements-kaggle-ocr.txt',
    'config/requirements-kaggle-ocr-py313.txt',
    'schemas/ocr-sidecar-v1.schema.json',
}


def canonical_json(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(',', ':'), allow_nan=False) + '\n').encode('utf-8')


def safe_member(name):
    if (not isinstance(name, str) or not name or '\\' in name or ':' in name
            or PurePosixPath(name).is_absolute()
            or any(ord(c) < 32 or ord(c) == 127 for c in name)
            or any(p in ('', '.', '..') or p.endswith(('.', ' '))
                   for p in name.split('/'))):
        raise ValueError('Unsafe relative POSIX payload path')
    if name not in ALLOWED_FILES:
        raise ValueError('Payload path is not allowlisted: ' + name)


def validate_payload(payload, expected_hash):
    if not isinstance(payload, dict) or set(payload) != {'format', 'files'}:
        raise ValueError('Malformed embedded payload')
    if payload['format'] != 'kis-kaggle-ocr-v5' or not isinstance(payload['files'], list):
        raise ValueError('Unsupported embedded payload')
    if len(payload['files']) != len(ALLOWED_FILES):
        raise ValueError('Payload allowlist membership mismatch')
    decoded = {}
    for entry in payload['files']:
        if not isinstance(entry, dict) or set(entry) != {'path', 'size', 'sha256', 'base64'}:
            raise ValueError('Malformed payload entry')
        name = entry['path']
        safe_member(name)
        if name in decoded:
            raise ValueError('Duplicate payload path')
        size, checksum, text = entry['size'], entry['sha256'], entry['base64']
        if (type(size) is not int or not 0 < size <= MAX_PAYLOAD_BYTES
                or not isinstance(checksum, str) or not re.fullmatch('[0-9a-f]{64}', checksum)
                or not isinstance(text, str) or len(text) > 4 * ((size + 2) // 3)):
            raise ValueError('Invalid payload size/hash/base64')
        data = base64.b64decode(text, validate=True)
        if (len(data) != size or base64.b64encode(data).decode('ascii') != text
                or hashlib.sha256(data).hexdigest() != checksum):
            raise ValueError('Payload bytes/hash/size mismatch: ' + name)
        if '\r' in data.decode('utf-8') or data.startswith(b'\xef\xbb\xbf') or not data.endswith(b'\n'):
            raise ValueError('Payload must use canonical UTF-8/LF')
        decoded[name] = data
    if set(decoded) != ALLOWED_FILES or list(decoded) != sorted(ALLOWED_FILES):
        raise ValueError('Payload allowlist/order mismatch')
    encoded = canonical_json(payload)
    if len(encoded) > MAX_PAYLOAD_BYTES or sum(map(len, decoded.values())) > MAX_PAYLOAD_BYTES:
        raise ValueError('Embedded payload exceeds 1 MiB')
    if not isinstance(expected_hash, str) or not re.fullmatch('[0-9a-f]{64}', expected_hash):
        raise ValueError('Malformed payload hash')
    if hashlib.sha256(encoded).hexdigest() != expected_hash:
        raise ValueError('Embedded payload hash mismatch')
    return decoded


def reject_links(path):
    # lstat also catches Windows junctions/reparse points on Python 3.10/3.11.
    path = Path(path).absolute()
    for part in (path, *path.parents):
        try:
            info = part.lstat()
        except FileNotFoundError:
            continue
        if (part.is_symlink() or stat.S_ISLNK(info.st_mode)
                or getattr(info, 'st_file_attributes', 0) & 0x400
                or (hasattr(part, 'is_junction') and part.is_junction())):
            raise ValueError('Payload/work symlink/junction/reparse point forbidden')
    return path


def materialize_payload(payload, expected_hash, working_base):
    files = validate_payload(payload, expected_hash)  # Validate ALL bytes before any write/import.
    base = reject_links(working_base)
    root = reject_links(base / expected_hash / 'code')
    if not root.resolve().is_relative_to(base.resolve()):
        raise ValueError('Payload target escapes working base')
    allowed_dirs = {str(PurePosixPath(n).parent) for n in files}
    if root.exists():
        for path in root.rglob('*'):
            reject_links(path)
            relative = path.relative_to(root).as_posix()
            if path.is_dir() and relative in allowed_dirs:
                continue
            if relative not in files or not path.is_file():
                raise ValueError('Unexpected materialized payload member')
    # Check all cached targets before creating/reusing any source file. Never overwrite tampering.
    for name, data in files.items():
        target = reject_links(root / name)
        if target.exists() and (not target.is_file() or target.stat().st_size != len(data)
                                or target.read_bytes() != data):
            raise ValueError('Materialized payload tamper: ' + name)
    root.mkdir(parents=True, exist_ok=True)
    for name, data in files.items():
        target = reject_links(root / name)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            with target.open('xb') as stream:
                stream.write(data)
        reject_links(target)
        if target.read_bytes() != data:
            raise ValueError('Materialized payload changed: ' + name)
    return root

EMBEDDED_PAYLOAD = json.loads('{"files":[{"base64":"ewogICJzY2hlbWEiOiAia2lzX2thZ2dsZV9vY3JfY29uZmlnX3YxIiwKICAib3V0cHV0X3NjaGVtYSI6ICJvY3Itc2lkZWNhci12MSIsCiAgImRldmljZSI6ICJjcHUiLAogICJwYWRkbGVfZGlzdHJpYnV0aW9uIjogInBhZGRsZXBhZGRsZSIsCiAgInZlcnNpb25zIjogewogICAgInBhZGRsZW9jciI6ICIzLjcuMCIsCiAgICAicGFkZGxleCI6ICIzLjcuMCIsCiAgICAicGFkZGxlcGFkZGxlIjogIjMuMy4xIiwKICAgICJudW1weSI6ICIyLjIuNiIsCiAgICAiUGlsbG93IjogIjExLjEuMCIsCiAgICAiaHVnZ2luZ2ZhY2UtaHViIjogIjEuMjcuMCIsCiAgICAianNvbnNjaGVtYSI6ICI0LjIzLjAiCiAgfSwKICAib2NyX3ZlcnNpb24iOiAiUFAtT0NSdjUiLAogICJsYW5ndWFnZSI6ICJsYXRpbiAoaW5jbHVkaW5nIFZpZXRuYW1lc2UpOyBub3QgYXV0by1kZXRlY3RlZCIsCiAgIm1vZGVscyI6IFsKICAgIHsKICAgICAgInJvbGUiOiAiZGV0ZWN0aW9uIiwKICAgICAgIm5hbWUiOiAiUFAtT0NSdjVfbW9iaWxlX2RldCIsCiAgICAgICJyZXBvIjogIlBhZGRsZVBhZGRsZS9QUC1PQ1J2NV9tb2JpbGVfZGV0IiwKICAgICAgInJldmlzaW9uIjogIjBkNjNlNzhlMmI2ODA5MjhmNmIxNzQ3ZDc2YTA4ZGI2ZTY0NWVmYjciCiAgICB9LAogICAgewogICAgICAicm9sZSI6ICJyZWNvZ25pdGlvbiIsCiAgICAgICJuYW1lIjogImxhdGluX1BQLU9DUnY1X21vYmlsZV9yZWMiLAogICAgICAicmVwbyI6ICJQYWRkbGVQYWRkbGUvbGF0aW5fUFAtT0NSdjVfbW9iaWxlX3JlYyIsCiAgICAgICJyZXZpc2lvbiI6ICJhYjJjZDVjYzVmYTYzMDliZTJlNWFjZGZlNjZlY2EyYzJjMTI3ZDU3IgogICAgfQogIF0sCiAgInBhZGRpbmciOiB7ImxlZnQiOiAzMiwgInRvcCI6IDMyLCAicmlnaHQiOiAzMiwgImJvdHRvbSI6IDMyLCAiY29sb3IiOiBbMjU1LCAyNTUsIDI1NV19LAogICJyZXNpemUiOiBudWxsLAogICJiYXRjaF9zaXplIjogMSwKICAiY2hlY2twb2ludF9ldmVyeSI6IDEsCiAgInRleHRfcmVjX3Njb3JlX3RocmVzaCI6IDAuMAp9Cg==","path":"config/ocr-kaggle-py313-v5.json","sha256":"ef49d0545486d4e5ccc97ef3b28dc18df99094ddee83bef9fca59e1cdd7ad141","size":1033},{"base64":"ewogICJzY2hlbWEiOiAia2lzX2thZ2dsZV9vY3JfY29uZmlnX3YxIiwKICAib3V0cHV0X3NjaGVtYSI6ICJvY3Itc2lkZWNhci12MSIsCiAgImRldmljZSI6ICJjcHUiLAogICJwYWRkbGVfZGlzdHJpYnV0aW9uIjogInBhZGRsZXBhZGRsZSIsCiAgInZlcnNpb25zIjogewogICAgInBhZGRsZW9jciI6ICIzLjcuMCIsCiAgICAicGFkZGxleCI6ICIzLjcuMCIsCiAgICAicGFkZGxlcGFkZGxlIjogIjMuMy4xIiwKICAgICJudW1weSI6ICIyLjIuNiIsCiAgICAiUGlsbG93IjogIjExLjEuMCIsCiAgICAiaHVnZ2luZ2ZhY2UtaHViIjogIjEuMjcuMCIsCiAgICAianNvbnNjaGVtYSI6ICI0LjIzLjAiCiAgfSwKICAib2NyX3ZlcnNpb24iOiAiUFAtT0NSdjUiLAogICJsYW5ndWFnZSI6ICJsYXRpbiAoaW5jbHVkaW5nIFZpZXRuYW1lc2UpOyBub3QgYXV0by1kZXRlY3RlZCIsCiAgIm1vZGVscyI6IFsKICAgIHsKICAgICAgInJvbGUiOiAiZGV0ZWN0aW9uIiwKICAgICAgIm5hbWUiOiAiUFAtT0NSdjVfbW9iaWxlX2RldCIsCiAgICAgICJyZXBvIjogIlBhZGRsZVBhZGRsZS9QUC1PQ1J2NV9tb2JpbGVfZGV0IiwKICAgICAgInJldmlzaW9uIjogIjBkNjNlNzhlMmI2ODA5MjhmNmIxNzQ3ZDc2YTA4ZGI2ZTY0NWVmYjciCiAgICB9LAogICAgewogICAgICAicm9sZSI6ICJyZWNvZ25pdGlvbiIsCiAgICAgICJuYW1lIjogImxhdGluX1BQLU9DUnY1X21vYmlsZV9yZWMiLAogICAgICAicmVwbyI6ICJQYWRkbGVQYWRkbGUvbGF0aW5fUFAtT0NSdjVfbW9iaWxlX3JlYyIsCiAgICAgICJyZXZpc2lvbiI6ICJhYjJjZDVjYzVmYTYzMDliZTJlNWFjZGZlNjZlY2EyYzJjMTI3ZDU3IgogICAgfQogIF0sCiAgInBhZGRpbmciOiB7ImxlZnQiOiAzMiwgInRvcCI6IDMyLCAicmlnaHQiOiAzMiwgImJvdHRvbSI6IDMyLCAiY29sb3IiOiBbMjU1LCAyNTUsIDI1NV19LAogICJyZXNpemUiOiBudWxsLAogICJiYXRjaF9zaXplIjogMSwKICAiY2hlY2twb2ludF9ldmVyeSI6IDEsCiAgInRleHRfcmVjX3Njb3JlX3RocmVzaCI6IDAuMAp9Cg==","path":"config/ocr-kaggle-py313.json","sha256":"ef49d0545486d4e5ccc97ef3b28dc18df99094ddee83bef9fca59e1cdd7ad141","size":1033},{"base64":"ewogICJzY2hlbWEiOiAia2lzX2thZ2dsZV9vY3JfY29uZmlnX3YxIiwKICAib3V0cHV0X3NjaGVtYSI6ICJvY3Itc2lkZWNhci12MSIsCiAgImRldmljZSI6ICJjcHUiLAogICJwYWRkbGVfZGlzdHJpYnV0aW9uIjogInBhZGRsZXBhZGRsZSIsCiAgInZlcnNpb25zIjogewogICAgInBhZGRsZW9jciI6ICIzLjcuMCIsCiAgICAicGFkZGxleCI6ICIzLjcuMCIsCiAgICAicGFkZGxlcGFkZGxlIjogIjMuMy4xIiwKICAgICJudW1weSI6ICIxLjI2LjQiLAogICAgIlBpbGxvdyI6ICIxMS4xLjAiLAogICAgImh1Z2dpbmdmYWNlLWh1YiI6ICIxLjI3LjAiLAogICAgImpzb25zY2hlbWEiOiAiNC4yMy4wIgogIH0sCiAgIm9jcl92ZXJzaW9uIjogIlBQLU9DUnY1IiwKICAibGFuZ3VhZ2UiOiAibGF0aW4gKGluY2x1ZGluZyBWaWV0bmFtZXNlKTsgbm90IGF1dG8tZGV0ZWN0ZWQiLAogICJtb2RlbHMiOiBbCiAgICB7CiAgICAgICJyb2xlIjogImRldGVjdGlvbiIsCiAgICAgICJuYW1lIjogIlBQLU9DUnY1X21vYmlsZV9kZXQiLAogICAgICAicmVwbyI6ICJQYWRkbGVQYWRkbGUvUFAtT0NSdjVfbW9iaWxlX2RldCIsCiAgICAgICJyZXZpc2lvbiI6ICIwZDYzZTc4ZTJiNjgwOTI4ZjZiMTc0N2Q3NmEwOGRiNmU2NDVlZmI3IgogICAgfSwKICAgIHsKICAgICAgInJvbGUiOiAicmVjb2duaXRpb24iLAogICAgICAibmFtZSI6ICJsYXRpbl9QUC1PQ1J2NV9tb2JpbGVfcmVjIiwKICAgICAgInJlcG8iOiAiUGFkZGxlUGFkZGxlL2xhdGluX1BQLU9DUnY1X21vYmlsZV9yZWMiLAogICAgICAicmV2aXNpb24iOiAiYWIyY2Q1Y2M1ZmE2MzA5YmUyZTVhY2RmZTY2ZWNhMmMyYzEyN2Q1NyIKICAgIH0KICBdLAogICJwYWRkaW5nIjogeyJsZWZ0IjogMzIsICJ0b3AiOiAzMiwgInJpZ2h0IjogMzIsICJib3R0b20iOiAzMiwgImNvbG9yIjogWzI1NSwgMjU1LCAyNTVdfSwKICAicmVzaXplIjogbnVsbCwKICAiYmF0Y2hfc2l6ZSI6IDEsCiAgImNoZWNrcG9pbnRfZXZlcnkiOiAxLAogICJ0ZXh0X3JlY19zY29yZV90aHJlc2giOiAwLjAKfQo=","path":"config/ocr-kaggle.json","sha256":"15209045d54efb2911c104117437819442b471cf8591e930179935cae445398f","size":1034},{"base64":"IyBQeXRob24gMy4xMyBDUFUgY2FuZGlkYXRlOyBub3QgbG9jYWwgaW5mZXJlbmNlIHZlcmlmaWVkLiBObyBzb3VyY2UgYnVpbGRzL3VucGlubmluZy4KIyBOdW1QeSAyLjIuNiBoYXMgQ1B5dGhvbiAzLjEzIHdoZWVsczsgMS4yNi40IGRvZXMgbm90LiBLZWVwIHZlcnNpb25zIGluIEpTT04gaW4gc3luYy4KIyBQYWRkbGVPQ1IgMy43LjAgcmVxdWlyZXMgcGFkZGxleD49My43LjAsPDMuOC4wOiBwaW4gdGhlIHNhbWUgbWlub3IgZXhhY3RseS4KIyBGdWxsIHRyYW5zaXRpdmUgd2hlZWwgY29tcGF0aWJpbGl0eSBtdXN0IHBhc3MgYmluYXJ5LW9ubHkgcmVzb2x2ZXIgKyBpbXBvcnQgc21va2Ugb24gS2FnZ2xlLgpwYWRkbGVvY3I9PTMuNy4wCnBhZGRsZXg9PTMuNy4wCnBhZGRsZXBhZGRsZT09My4zLjEKbnVtcHk9PTIuMi42Cmh1Z2dpbmdmYWNlLWh1Yj09MS4yNy4wClBpbGxvdz09MTEuMS4wCmpzb25zY2hlbWE9PTQuMjMuMAo=","path":"config/requirements-kaggle-ocr-py313.txt","sha256":"fb73dd8388b582617c6373d48de999ed334d545d9fbf13416a7202c01cab4a17","size":470},{"base64":"IyBJc29sYXRlZCBQeXRob24gMy4xMC0zLjEyIGVudmlyb25tZW50IE9OTFk7IGluaGVyaXRlZCBmcm9tIHByZXBhcmVfb2NyX3BpbG90LnB5LgojIFRoZXNlIHBpbnMgYXJlIG5vdCBwcm9vZiB0aGF0IHdoZWVscyBleGlzdCBmb3IgZXZlcnkgS2FnZ2xlIGltYWdlL0NVREEgcnVudGltZS4KIyBTdG9wIGlmIHVuYXZhaWxhYmxlOyBkbyBub3QgdW5waW4gb3IgZG93bmdyYWRlIHRoZSByZXBvc2l0b3J5IGVudmlyb25tZW50LgpwYWRkbGVvY3I9PTMuNy4wCnBhZGRsZXg9PTMuNy4wCnBhZGRsZXBhZGRsZT09My4zLjEKbnVtcHk9PTEuMjYuNApodWdnaW5nZmFjZS1odWI9PTEuMjcuMApQaWxsb3c9PTExLjEuMApqc29uc2NoZW1hPT00LjIzLjAK","path":"config/requirements-kaggle-ocr.txt","sha256":"96fdc2c05f4b4e3734f67ab6d734983b814d922134be4fc75498b9b7bd7a510d","size":366},{"base64":"ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogInVybjpraXM6b2NyLXNpZGVjYXItdjEiLAogICJ0aXRsZSI6ICJJbmRlcGVuZGVudCBPQ1IgSlNPTkwgcmVjb3JkLCBjYW5vbmljYWwga2V5ICh2aWRlb19pZCwga2V5ZnJhbWVfbikiLAogICJkZXNjcmlwdGlvbiI6ICJPbmUgZW52ZWxvcGUgcGVyIEpTT05MIGxpbmUuIG1hbmlmZXN0Lmpzb24gcmVwZWF0cyBzY2hlbWEvc2NoZW1hX3ZlcnNpb24gYW5kIGludmVudG9yaWVzIGFsbCBjYW5vbmljYWwga2V5cy4gUnVudGltZSB2YWxpZGF0aW9uIGFsc28gY2hlY2tzIHVuaXF1ZSBrZXlzLCBoYXNoZXMsIHRyYW5zZm9ybXMsIGxpbmUgc29ydGluZyBhbmQgc3RhdHVzIHNlbWFudGljcy4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWyJzY2hlbWEiLCAic2NoZW1hX3ZlcnNpb24iLCAiZ3JvdXAiLCAidmlkZW9faWQiLCAia2V5ZnJhbWVfbiIsICJpbWFnZV9uYW1lIiwgInNvdXJjZV9wYXRoIiwgImltYWdlX3NoYTI1NiIsICJtYXBwaW5nIiwgImVuZ2luZSIsICJzdGF0dXMiLCAicHJlcHJvY2Vzc2luZyIsICJsaW5lcyIsICJlcnJvciJdLAogICJwcm9wZXJ0aWVzIjogewogICAgInNjaGVtYSI6IHsiY29uc3QiOiAib2NyLXNpZGVjYXItdjEifSwKICAgICJzY2hlbWFfdmVyc2lvbiI6IHsiY29uc3QiOiAxfSwKICAgICJncm91cCI6IHsidHlwZSI6ICJzdHJpbmciLCAicGF0dGVybiI6ICJeTFswLTldKyQifSwKICAgICJ2aWRlb19pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAicGF0dGVybiI6ICJeTFswLTldK19WWzAtOV0rJCJ9LAogICAgImtleWZyYW1lX24iOiB7InR5cGUiOiAiaW50ZWdlciIsICJtaW5pbXVtIjogMH0sCiAgICAiaW1hZ2VfbmFtZSI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXSwgInBhdHRlcm4iOiAiXlswLTldK1xcLihbakpdW3BQXVtlRV0/W2dHXXxbcFBdW25OXVtnR10pJCJ9LAogICAgInNvdXJjZV9wYXRoIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdLCAibWluTGVuZ3RoIjogMX0sCiAgICAiaW1hZ2Vfc2hhMjU2IjogeyJhbnlPZiI6IFt7IiRyZWYiOiAiIy8kZGVmcy9zaGEyNTYifSwgeyJ0eXBlIjogIm51bGwifV19LAogICAgIm1hcHBpbmciOiB7CiAgICAgICJ0eXBlIjogWyJvYmplY3QiLCAibnVsbCJdLAogICAgICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBmYWxzZSwKICAgICAgInJlcXVpcmVkIjogWyJmcmFtZV9pZHgiLCAidGltZXN0YW1wX3NlY29uZHMiXSwKICAgICAgInByb3BlcnRpZXMiOiB7CiAgICAgICAgImZyYW1lX2lkeCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAwfSwKICAgICAgICAidGltZXN0YW1wX3NlY29uZHMiOiB7InR5cGUiOiAibnVtYmVyIiwgIm1pbmltdW0iOiAwfQogICAgICB9CiAgICB9LAogICAgImVuZ2luZSI6IHsiJHJlZiI6ICIjLyRkZWZzL2VuZ2luZSJ9LAogICAgInN0YXR1cyI6IHsiZW51bSI6IFsib2siLCAiZW1wdHkiLCAibWlzc2luZyIsICJlcnJvciIsICJ1bmtub3duIl19LAogICAgInByZXByb2Nlc3NpbmciOiB7ImFueU9mIjogW3siJHJlZiI6ICIjLyRkZWZzL3ByZXByb2Nlc3NpbmcifSwgeyJ0eXBlIjogIm51bGwifV19LAogICAgImxpbmVzIjogeyJ0eXBlIjogImFycmF5IiwgIml0ZW1zIjogeyIkcmVmIjogIiMvJGRlZnMvbGluZSJ9fSwKICAgICJlcnJvciI6IHsKICAgICAgInR5cGUiOiBbIm9iamVjdCIsICJudWxsIl0sCiAgICAgICJhZGRpdGlvbmFsUHJvcGVydGllcyI6IGZhbHNlLAogICAgICAicmVxdWlyZWQiOiBbInR5cGUiLCAiY29kZSJdLAogICAgICAicHJvcGVydGllcyI6IHsidHlwZSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sICJjb2RlIjogeyJjb25zdCI6ICJPQ1JfRkFJTEVEIn19CiAgICB9CiAgfSwKICAiYWxsT2YiOiBbCiAgICB7ImlmIjogeyJwcm9wZXJ0aWVzIjogeyJzdGF0dXMiOiB7ImNvbnN0IjogIm9rIn19fSwgInRoZW4iOiB7InByb3BlcnRpZXMiOiB7ImxpbmVzIjogeyJtaW5JdGVtcyI6IDF9fX0sICJlbHNlIjogeyJwcm9wZXJ0aWVzIjogeyJsaW5lcyI6IHsibWF4SXRlbXMiOiAwfX19fSwKICAgIHsiaWYiOiB7InByb3BlcnRpZXMiOiB7InN0YXR1cyI6IHsiY29uc3QiOiAiZXJyb3IifX19LCAidGhlbiI6IHsicHJvcGVydGllcyI6IHsiZXJyb3IiOiB7InR5cGUiOiAib2JqZWN0In19fSwgImVsc2UiOiB7InByb3BlcnRpZXMiOiB7ImVycm9yIjogeyJ0eXBlIjogIm51bGwifX19fSwKICAgIHsiaWYiOiB7InByb3BlcnRpZXMiOiB7InN0YXR1cyI6IHsiZW51bSI6IFsib2siLCAiZW1wdHkiXX19fSwgInRoZW4iOiB7InByb3BlcnRpZXMiOiB7InByZXByb2Nlc3NpbmciOiB7InR5cGUiOiAib2JqZWN0In0sICJpbWFnZV9zaGEyNTYiOiB7IiRyZWYiOiAiIy8kZGVmcy9zaGEyNTYifSwgInNvdXJjZV9wYXRoIjogeyJ0eXBlIjogInN0cmluZyJ9LCAiaW1hZ2VfbmFtZSI6IHsidHlwZSI6ICJzdHJpbmcifX19fQogIF0sCiAgIiRkZWZzIjogewogICAgInNoYTI1NiI6IHsidHlwZSI6ICJzdHJpbmciLCAicGF0dGVybiI6ICJeWzAtOWEtZl17NjR9JCJ9LAogICAgInNpemUiOiB7InR5cGUiOiAiYXJyYXkiLCAicHJlZml4SXRlbXMiOiBbeyJ0eXBlIjogImludGVnZXIiLCAibWluaW11bSI6IDF9LCB7InR5cGUiOiAiaW50ZWdlciIsICJtaW5pbXVtIjogMX1dLCAiaXRlbXMiOiBmYWxzZSwgIm1pbkl0ZW1zIjogMn0sCiAgICAicG9pbnQiOiB7InR5cGUiOiAiYXJyYXkiLCAiaXRlbXMiOiB7InR5cGUiOiAibnVtYmVyIn0sICJtaW5JdGVtcyI6IDIsICJtYXhJdGVtcyI6IDJ9LAogICAgInBvbHlnb24iOiB7InR5cGUiOiAiYXJyYXkiLCAiaXRlbXMiOiB7IiRyZWYiOiAiIy8kZGVmcy9wb2ludCJ9LCAibWluSXRlbXMiOiAzfSwKICAgICJiYm94IjogeyJ0eXBlIjogImFycmF5IiwgIml0ZW1zIjogeyJ0eXBlIjogIm51bWJlciJ9LCAibWluSXRlbXMiOiA0LCAibWF4SXRlbXMiOiA0fSwKICAgICJtYXRyaXgiOiB7InR5cGUiOiAiYXJyYXkiLCAibWluSXRlbXMiOiAzLCAibWF4SXRlbXMiOiAzLCAiaXRlbXMiOiB7InR5cGUiOiAiYXJyYXkiLCAibWluSXRlbXMiOiAzLCAibWF4SXRlbXMiOiAzLCAiaXRlbXMiOiB7InR5cGUiOiAibnVtYmVyIn19fSwKICAgICJwYWRkaW5nIjogewogICAgICAidHlwZSI6ICJvYmplY3QiLCAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBmYWxzZSwKICAgICAgInJlcXVpcmVkIjogWyJsZWZ0IiwgInRvcCIsICJyaWdodCIsICJib3R0b20iLCAiY29sb3IiXSwKICAgICAgInByb3BlcnRpZXMiOiB7CiAgICAgICAgImxlZnQiOiB7InR5cGUiOiAiaW50ZWdlciIsICJtaW5pbXVtIjogMCwgIm1heGltdW0iOiA0MDk2fSwKICAgICAgICAidG9wIjogeyJ0eXBlIjogImludGVnZXIiLCAibWluaW11bSI6IDAsICJtYXhpbXVtIjogNDA5Nn0sCiAgICAgICAgInJpZ2h0IjogeyJ0eXBlIjogImludGVnZXIiLCAibWluaW11bSI6IDAsICJtYXhpbXVtIjogNDA5Nn0sCiAgICAgICAgImJvdHRvbSI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAwLCAibWF4aW11bSI6IDQwOTZ9LAogICAgICAgICJjb2xvciI6IHsidHlwZSI6ICJhcnJheSIsICJtaW5JdGVtcyI6IDMsICJtYXhJdGVtcyI6IDMsICJpdGVtcyI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAwLCAibWF4aW11bSI6IDI1NX19CiAgICAgIH0KICAgIH0sCiAgICAicHJlcHJvY2Vzc2luZyI6IHsKICAgICAgInR5cGUiOiAib2JqZWN0IiwgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgICAgICJyZXF1aXJlZCI6IFsicGFkZGluZyIsICJvcmlnaW5hbF9zaXplIiwgInBhZGRlZF9zaXplIiwgInJlc2l6ZSIsICJmb3J3YXJkX3RyYW5zZm9ybSIsICJpbnZlcnNlX3RyYW5zZm9ybSIsICJjb29yZGluYXRlX3NwYWNlIl0sCiAgICAgICJwcm9wZXJ0aWVzIjogewogICAgICAgICJwYWRkaW5nIjogeyIkcmVmIjogIiMvJGRlZnMvcGFkZGluZyJ9LAogICAgICAgICJvcmlnaW5hbF9zaXplIjogeyIkcmVmIjogIiMvJGRlZnMvc2l6ZSJ9LAogICAgICAgICJwYWRkZWRfc2l6ZSI6IHsiJHJlZiI6ICIjLyRkZWZzL3NpemUifSwKICAgICAgICAicmVzaXplIjogeyJ0eXBlIjogIm51bGwifSwKICAgICAgICAiZm9yd2FyZF90cmFuc2Zvcm0iOiB7IiRyZWYiOiAiIy8kZGVmcy9tYXRyaXgifSwKICAgICAgICAiaW52ZXJzZV90cmFuc2Zvcm0iOiB7IiRyZWYiOiAiIy8kZGVmcy9tYXRyaXgifSwKICAgICAgICAiY29vcmRpbmF0ZV9zcGFjZSI6IHsiY29uc3QiOiAiUGFkZGxlIHByZWRpY3QgcmVjX3BvbHlzIG1hcHBlZCBiYWNrIHRvIHN1cHBsaWVkIHBhZGRlZCBpbWFnZTsgbm90IG1vZGVsIHRlbnNvcnMifQogICAgICB9CiAgICB9LAogICAgImVuZ2luZSI6IHsKICAgICAgInR5cGUiOiAib2JqZWN0IiwgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgICAgICJyZXF1aXJlZCI6IFsibmFtZSIsICJ2ZXJzaW9ucyIsICJtb2RlbHMiLCAiZGV2aWNlIiwgImxhbmd1YWdlIiwgImNvbmZpZ19zaGEyNTYiXSwKICAgICAgInByb3BlcnRpZXMiOiB7CiAgICAgICAgIm5hbWUiOiB7ImNvbnN0IjogInBhZGRsZW9jciJ9LAogICAgICAgICJ2ZXJzaW9ucyI6IHsidHlwZSI6ICJvYmplY3QiLCAibWluUHJvcGVydGllcyI6IDEsICJhZGRpdGlvbmFsUHJvcGVydGllcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX19LAogICAgICAgICJkZXZpY2UiOiB7ImVudW0iOiBbImNwdSIsICJncHU6MCJdfSwKICAgICAgICAibGFuZ3VhZ2UiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgICAgICJjb25maWdfc2hhMjU2IjogeyIkcmVmIjogIiMvJGRlZnMvc2hhMjU2In0sCiAgICAgICAgIm1vZGVscyI6IHsKICAgICAgICAgICJ0eXBlIjogImFycmF5IiwgIm1pbkl0ZW1zIjogMiwgIm1heEl0ZW1zIjogMiwKICAgICAgICAgICJpdGVtcyI6IHsKICAgICAgICAgICAgInR5cGUiOiAib2JqZWN0IiwgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgICAgICAgICAgICJyZXF1aXJlZCI6IFsicm9sZSIsICJuYW1lIiwgInJlcG8iLCAicmV2aXNpb24iLCAiZmlsZXNfc2hhMjU2Il0sCiAgICAgICAgICAgICJwcm9wZXJ0aWVzIjogewogICAgICAgICAgICAgICJyb2xlIjogeyJlbnVtIjogWyJkZXRlY3Rpb24iLCAicmVjb2duaXRpb24iXX0sCiAgICAgICAgICAgICAgIm5hbWUiOiB7InR5cGUiOiAic3RyaW5nIiwgInBhdHRlcm4iOiAiXltBLVphLXowLTlfLV0rJCJ9LAogICAgICAgICAgICAgICJyZXBvIjogeyJ0eXBlIjogInN0cmluZyIsICJwYXR0ZXJuIjogIl5bQS1aYS16MC05Xy1dKy9bQS1aYS16MC05Xy1dKyQifSwKICAgICAgICAgICAgICAicmV2aXNpb24iOiB7InR5cGUiOiAic3RyaW5nIiwgInBhdHRlcm4iOiAiXlswLTlhLWZdezQwfSQifSwKICAgICAgICAgICAgICAiZmlsZXNfc2hhMjU2IjogeyJ0eXBlIjogIm9iamVjdCIsICJyZXF1aXJlZCI6IFsiaW5mZXJlbmNlLmpzb24iLCAiaW5mZXJlbmNlLnBkaXBhcmFtcyIsICJpbmZlcmVuY2UueW1sIl0sICJhZGRpdGlvbmFsUHJvcGVydGllcyI6IHsiJHJlZiI6ICIjLyRkZWZzL3NoYTI1NiJ9fQogICAgICAgICAgICB9CiAgICAgICAgICB9CiAgICAgICAgfQogICAgICB9CiAgICB9LAogICAgImxpbmUiOiB7CiAgICAgICJ0eXBlIjogIm9iamVjdCIsICJhZGRpdGlvbmFsUHJvcGVydGllcyI6IGZhbHNlLAogICAgICAicmVxdWlyZWQiOiBbImxpbmVfaW5kZXgiLCAidGV4dCIsICJjb25maWRlbmNlIiwgImxhbmd1YWdlIiwgInNjcmlwdCIsICJwb2x5Z29uX3BhZGRlZCIsICJiYm94X3BhZGRlZCIsICJwb2x5Z29uX29yaWdpbmFsIiwgImJib3hfb3JpZ2luYWwiLCAicG9seWdvbl9vcmlnaW5hbF9jbGlwcGVkIiwgImJib3hfb3JpZ2luYWxfY2xpcHBlZCJdLAogICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAibGluZV9pbmRleCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAwfSwKICAgICAgICAidGV4dCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAgICAgImNvbmZpZGVuY2UiOiB7InR5cGUiOiAibnVtYmVyIiwgIm1pbmltdW0iOiAwLCAibWF4aW11bSI6IDF9LAogICAgICAgICJsYW5ndWFnZSI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAgICAgInNjcmlwdCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAgICAgInBvbHlnb25fcGFkZGVkIjogeyIkcmVmIjogIiMvJGRlZnMvcG9seWdvbiJ9LAogICAgICAgICJiYm94X3BhZGRlZCI6IHsiJHJlZiI6ICIjLyRkZWZzL2Jib3gifSwKICAgICAgICAicG9seWdvbl9vcmlnaW5hbCI6IHsiJHJlZiI6ICIjLyRkZWZzL3BvbHlnb24ifSwKICAgICAgICAiYmJveF9vcmlnaW5hbCI6IHsiJHJlZiI6ICIjLyRkZWZzL2Jib3gifSwKICAgICAgICAicG9seWdvbl9vcmlnaW5hbF9jbGlwcGVkIjogeyIkcmVmIjogIiMvJGRlZnMvcG9seWdvbiJ9LAogICAgICAgICJiYm94X29yaWdpbmFsX2NsaXBwZWQiOiB7IiRyZWYiOiAiIy8kZGVmcy9iYm94In0KICAgICAgfQogICAgfQogIH0KfQo=","path":"schemas/ocr-sidecar-v1.schema.json","sha256":"b45b159b9e919f811016fc16f0d1144264913d4afed58df2b878a85245c60cdf","size":6650},{"base64":"...","path":"tools/kaggle_ocr.py","sha256":"f6caa79f08660b942f31a6b1a45e29023518df2082d2b2735e732a8e21c984f9","size":48849},{"base64":"...","path":"tools/kaggle_ocr_v5.py","sha256":"61a8a99198511342af4e916daa0306f6f357adceeb77ad2c4e3ccf8249b1d271","size":44178}],"format":"kis-kaggle-ocr-v5"}\n')
PAYLOAD_SHA256 = '18404c1a92d90cf669dfad464412ba3f97f71b2552621a959ac7053995e65c64'
RUNTIME_READY = False
OCR_RUN_OK = False
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('DEPENDENCY_UNAVAILABLE: supported Python 3.10-3.13; ' + json.dumps({
        'python': sys.version, 'executable': sys.executable,
        'platform': sys.platform, 'profile': 'unsupported'}))
root = materialize_payload(EMBEDDED_PAYLOAD, PAYLOAD_SHA256,
                           Path('/kaggle/working/kis-kaggle-ocr-v5'))
work = root.parent
script = root / 'tools/kaggle_ocr_v5.py'
spec = importlib.util.spec_from_file_location('portable_kaggle_ocr_v5', script)
ocr = importlib.util.module_from_spec(spec)
previous_bytecode_setting = sys.dont_write_bytecode
try:
    sys.dont_write_bytecode = True  # Keep the materialized code tree limited to eight files.
    spec.loader.exec_module(ocr)
finally:
    sys.dont_write_bytecode = previous_bytecode_setting
profile = ocr.select_profile()
config_path = root / profile['config']
print(json.dumps({'payload_sha256': PAYLOAD_SHA256,
                  'interpreter': ocr.interpreter_fingerprint(), 'profile': profile}, indent=2))
subprocess.run([sys.executable, '-B', str(script), 'validate-input',
                '--input-root', INPUT_ROOT, '--config', str(config_path),
                '--diagnostics-root', str(work / 'outputs/diagnostics-v5')],
               env=ocr.runtime_environment(), check=True)
print('Embedded integrity and input layout validated. No package Dataset or OCR model used.')


In [ ]:
import os
import shutil
RUNTIME_READY = False
OCR_RUN_OK = False
wheelhouses = [p for p in Path('/kaggle/input').rglob('ocr-wheelhouse') if p.is_dir()]
if len(wheelhouses) > 1:
    raise ValueError('Attach at most one ocr-wheelhouse')
models = [p for p in Path('/kaggle/input').rglob('ocr-models') if p.is_dir()]
if len(models) > 1:
    raise ValueError('Attach at most one ocr-models folder')
reject_links(work / 'venv')
wheelhouse = reject_links(wheelhouses[0]) if wheelhouses else None


def pick_venv_root(candidates=('/kaggle/temp', '/tmp'), min_free_gb=8):
    # The venv (several GB with the GPU wheel) must not land in /kaggle/working, which Kaggle saves as output.
    mounts = [line.split() for line in Path('/proc/mounts').read_text().splitlines() if line.strip()] if Path('/proc/mounts').is_file() else []
    for candidate in candidates:
        base = Path(candidate)
        if not base.is_dir() or not os.access(base, os.W_OK | os.X_OK):
            continue
        mount = max((m for m in mounts if len(m) > 3 and str(base).startswith(m[1])), key=lambda m: len(m[1]), default=None)
        if mount and 'noexec' in mount[3].split(','):
            continue
        if shutil.disk_usage(base).free < min_free_gb * 1024 ** 3:
            continue
        return base.absolute() / 'kis-ocr-runtime' / PAYLOAD_SHA256[:16]
    return None


venv_root = pick_venv_root()
print('Runtime venv location:', (venv_root / 'venv') if venv_root else str(work / 'venv') + '  (no suitable scratch dir; this WILL be saved with the output)')


def make_runtime_env(python_path):
    env = ocr.runtime_environment(python_path, sys.executable)
    env.pop('KIS_OCR_MODEL_ROOT', None)
    env.pop('KIS_OCR_OFFLINE', None)
    if models:
        env['KIS_OCR_MODEL_ROOT'] = str(reject_links(models[0]))
        env['KIS_OCR_OFFLINE'] = '1'
    return env


def setup_runtime(device_plan):
    ocr.apply_device_plan(device_plan)  # also read by the child processes
    return ocr.prepare_runtime(work, wheelhouse, venv_root)


def keep_gpu_diagnostics():
    source = work / 'outputs/diagnostics-v5'
    if source.is_dir():
        shutil.copytree(source, work / 'outputs/diagnostics-v5-gpu-failed', dirs_exist_ok=True)


# GPU is used automatically when the Kaggle accelerator is a GPU; otherwise CPU.
# To force CPU, run  os.environ['KIS_OCR_DEVICE'] = 'cpu'  before this cell.
plan = ocr.plan_device()
print('Device plan:', json.dumps(plan, indent=2))
try:
    python, config_path, identity = setup_runtime(plan)
except Exception as exc:
    if plan['device'] != 'gpu':
        raise
    print('GPU RUNTIME SETUP FAILED; falling back to CPU. Reason:\n' + ocr.safe_text(exc, 2000))
    keep_gpu_diagnostics()
    plan = ocr.plan_device(force_cpu=True)
    python, config_path, identity = setup_runtime(plan)
print(json.dumps(identity, indent=2))
runtime_env = make_runtime_env(python)
RUNTIME_READY = True
print('Selected device:', plan['device'].upper(), '-', plan['reason'])
print('venv-host-pip: binary resolver, pip check and import smoke passed; no model initialized.')
print('Backend API has NOT yet been inference-verified. Run cell performs mandatory one-image smoke.')
print('A sitecustomize/wrapt warning alone is neither success nor failure. Do not install wrapt globally.')


In [ ]:
OCR_RUN_OK = False
if not globals().get('RUNTIME_READY', False):
    raise RuntimeError('DEPENDENCY_UNAVAILABLE: dependency gate must pass before output')
output = reject_links(work / 'outputs/ocr-v1')
if output.exists():
    for path in output.rglob('*'):
        reject_links(path)
diagnostics = output.parent / 'diagnostics-v5'
print('Current OCR diagnostics:', diagnostics / 'run-fatal.json')


def run_child():
    command = [str(python), '-B', str(script), 'run', '--input-root', INPUT_ROOT,
               '--output-root', str(output), '--config', str(config_path)]
    if (output / 'manifest.json').exists():
        command += ['--resume']  # Matching-only resume; no silent error retry.
    done = subprocess.run(command, env=runtime_env, cwd=root, check=False,
                          capture_output=True, text=True, encoding='utf-8', errors='replace')
    print('OCR child stdout:\n' + ocr.safe_text(done.stdout or ''))
    print('OCR child stderr:\n' + ocr.safe_text(done.stderr or ''), file=sys.stderr)
    return done


def gpu_smoke_failed():
    # Only a failed smoke (no output/checkpoint yet) whose own error points at the GPU/CUDA stack.
    failed = diagnostics / 'failed.json'
    if (output / 'manifest.json').exists() or not failed.is_file():
        return False
    for attempt in json.loads(failed.read_text(encoding='utf-8')).get('attempts', []):
        error = attempt.get('error') or {}
        if error.get('phase') == 'one-image-inference' or re.search(
                r'gpu|cuda|cudnn|cublas|nvidia|compute capability|out of memory',
                str(error.get('message', '')), re.I):
            return True
    return False


completed = run_child()
if completed.returncode != 0 and plan['device'] == 'gpu' and gpu_smoke_failed():
    print('GPU SMOKE FAILED; falling back to CPU (runtime venv is rebuilt with the CPU wheel).')
    shutil.copytree(diagnostics, output.parent / 'diagnostics-v5-gpu-failed', dirs_exist_ok=True)
    ((venv_root or work) / 'venv' / 'requirements.sha256').unlink(missing_ok=True)  # force a clean CPU venv
    plan = ocr.plan_device(force_cpu=True)
    python, config_path, identity = setup_runtime(plan)
    runtime_env = make_runtime_env(python)
    completed = run_child()
if completed.returncode != 0:
    raise RuntimeError('OCR_STOPPED: child exit ' + str(completed.returncode) + '; inspect ' + str(diagnostics / 'run-fatal.json') + '. Backend failures are fatal, not empty OCR.')
subprocess.run([str(python), '-B', str(script), 'validate-output', '--input-root', INPUT_ROOT,
                '--output-root', str(output)], cwd=root, env=runtime_env, check=True)
coverage = json.loads((output / 'coverage.json').read_text(encoding='utf-8'))
print(json.dumps(coverage, indent=2))
meta = json.loads((output / 'run_metadata.json').read_text(encoding='utf-8'))
print('OCR device:', meta['engine']['device'], '| engine versions:', json.dumps(meta['engine']['versions']))
if not coverage['complete']:
    print('INCOMPLETE: preserve missing/error/unknown. Explicit retry: --resume --retry-errors.')
OCR_RUN_OK = True


## Download
Review coverage; download both ZIP and SHA256 links.


In [ ]:
if not (globals().get('RUNTIME_READY', False) and globals().get('OCR_RUN_OK', False)):
    raise RuntimeError('DEPENDENCY_UNAVAILABLE: no output package before successful run/validation')
if 'output' not in globals() or not (output / 'manifest.json').is_file():
    raise RuntimeError('OCR_STOPPED: no current validated output to package')
run_hash = hashlib.sha256((output / 'manifest.json').read_bytes()).hexdigest()[:16]
archive = reject_links(work / ('ocr-sidecar-v1-' + run_hash + '.zip'))
if not archive.exists():
    subprocess.run([str(python), '-B', str(script), 'package', '--input-root', str(output),
                    '--output-root', str(archive)], cwd=root, env=runtime_env, check=True)
else:
    ocr.verify_zip(archive)
    with zipfile.ZipFile(archive) as zipped:
        if zipped.read('manifest.json') != (output / 'manifest.json').read_bytes():
            raise ValueError('Existing ZIP belongs to a different run')
checksum = hashlib.sha256(archive.read_bytes()).hexdigest()
checksum_path = reject_links(Path(str(archive) + '.sha256'))
checksum_path.write_bytes((checksum + '  ' + archive.name + '\n').encode('utf-8'))
print('ZIP SHA256:', checksum, 'bytes:', archive.stat().st_size)
from IPython.display import FileLink, display
display(FileLink(str(archive)))
display(FileLink(str(checksum_path)))
